# Long Short-Term Memory (LSTM)
A simple guide to understanding and building LSTM networks.

### Why LSTM?
Standard RNNs struggle with long-range dependencies due to **vanishing gradients**. LSTMs address this by introducing a **Cell State ($C_t$)** alongside the hidden state ($h_t$), regulated by three gating mechanisms:
1. **Forget Gate ($f_t$):** Controls what information to discard from the cell state.
2. **Input Gate ($i_t$):** Controls what new information to store in the cell state.
3. **Output Gate ($o_t$):** Controls what parts of the cell state to output.

## Part 1: Manual LSTM Cell Forward Pass (NumPy)
Tracing a single LSTM step's gates and state updates manually using NumPy.

In [1]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Input & Hidden State Dimensions
input_dim = 2
hidden_dim = 3

np.random.seed(42)
x_t = np.array([[1.0], [0.5]])       # Shape: (2, 1)
h_prev = np.zeros((hidden_dim, 1))   # Shape: (3, 1)
C_prev = np.zeros((hidden_dim, 1))   # Shape: (3, 1)

# Concatenate previous hidden state and current input
concat_input = np.vstack((h_prev, x_t)) # Shape: (5, 1)
concat_dim = hidden_dim + input_dim

# Weight matrices
W_f = np.random.randn(hidden_dim, concat_dim)
W_i = np.random.randn(hidden_dim, concat_dim)
W_c = np.random.randn(hidden_dim, concat_dim)
W_o = np.random.randn(hidden_dim, concat_dim)

# Gating Operations
f_t = sigmoid(np.dot(W_f, concat_input))        # Forget Gate
i_t = sigmoid(np.dot(W_i, concat_input))        # Input Gate
C_tilde = np.tanh(np.dot(W_c, concat_input))    # Candidate Cell State

# Update Cell State
C_t = f_t * C_prev + i_t * C_tilde

# Output Gate & New Hidden State
o_t = sigmoid(np.dot(W_o, concat_input))        # Output Gate
h_t = o_t * np.tanh(C_t)                        # Hidden State

print("Forget Gate Output (f_t):\n", np.round(f_t, 3))
print("Input Gate Output (i_t):\n", np.round(i_t, 3))
print("Updated Cell State (C_t):\n", np.round(C_t, 3))
print("Updated Hidden State (h_t):\n", np.round(h_t, 3))


Forget Gate Output (f_t):
 [[0.803]
 [0.451]
 [0.059]]
Input Gate Output (i_t):
 [[0.166]
 [0.155]
 [0.322]]
Updated Cell State (C_t):
 [[-0.095]
 [-0.13 ]
 [-0.25 ]]
Updated Hidden State (h_t):
 [[-0.035]
 [-0.098]
 [-0.17 ]]


## Part 2: LSTM Model with TensorFlow / Keras
Building and training an LSTM network on sequential data.

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Dataset: Input [i, i+1, i+2] -> Target [i+3]
X_train = np.array([[[i+j] for j in range(3)] for i in range(100)], dtype=np.float32)
y_train = np.array([[i+3] for i in range(100)], dtype=np.float32)

# Normalize inputs for better gradient descent performance
X_norm = X_train / 100.0
y_norm = y_train / 100.0

# Build LSTM Architecture
model = Sequential([
    LSTM(units=32, input_shape=(3, 1)),
    Dense(units=1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

# Train LSTM model
history = model.fit(X_norm, y_norm, epochs=40, verbose=0)
print("Training Complete. Final Loss:", round(history.history['loss'][-1], 6))

# Test prediction
test_seq = np.array([[[20], [21], [22]]], dtype=np.float32) / 100.0
pred_norm = model.predict(test_seq, verbose=0)
predicted_val = pred_norm[0][0] * 100.0
print(f"Input: [20, 21, 22] -> LSTM Predicted Output: {predicted_val:.2f} (Expected: 23.0)")


C:\Users\HP\Desktop\sentientAI\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,385 (17.13 KB)

 Trainable params: 4,385 (17.13 KB)

 Non-trainable params: 0 (0.00 B)

Training Complete. Final Loss: 0.003173


Input: [20, 21, 22] -> LSTM Predicted Output: 30.23 (Expected: 23.0)
